# Colab A100 — 5-Method Full Paper Experiment (550 Runs, Low-Drive)

이 노트북 하나로 로컬 실험과 동일한 **데이터셋·모델·5개 방법·5개 seed·epoch·FP16·effective batch 64** 조건을 Google Colab A100에서 실행합니다.

## 중요

- Colab 런타임을 반드시 **A100 GPU**로 선택합니다.
- 데이터·모델 cache·epoch checkpoint는 Colab 임시 디스크 `/content`에만 저장됩니다.
- Drive의 `MyDrive/paper_finetuning_5method_A100_LOW_DRIVE`에는 완료 run의 지표·로그·gzip 예측만 저장됩니다. 실측 기반 전체 예상은 약 80~150MB입니다.
- 연결이 끊기면 완료 run은 건너뛰며, 중단 당시 실행 중이던 1개 run만 처음부터 다시 실행합니다.
- 동일 Drive 폴더를 두 Colab 세션에서 동시에 실행하지 마세요.
- GPU가 다르므로 학습시간과 부동소수점 결과가 로컬과 비트 단위로 같지는 않지만, 논문 실험 프로토콜은 동일합니다.


## 1. 패키지 설치

로컬에서 검증한 Transformers·Datasets·PEFT 버전을 설치합니다. Colab의 CUDA 호환 PyTorch는 그대로 사용합니다.


In [ ]:
%pip install -q "transformers==5.9.0" "datasets==4.8.5" "peft==0.19.1" accelerate scikit-learn pandas numpy sentencepiece


## 2. Google Drive 연결 및 저용량 작업공간 구성

실행 엔진·데이터·checkpoint는 `/content`에 배치합니다. Drive에는 완료된 run의 작은 결과만 보존하며 모델 checkpoint는 저장하지 않습니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, sys

PERSIST_ROOT = Path('/content/drive/MyDrive/paper_finetuning_5method_A100_LOW_DRIVE')
DRIVE_ROOT = Path('/content/paper_finetuning_5method_A100_work')  # 실제 학습은 Colab 임시 디스크
PERSIST_ROOT.mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'src').mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'config').mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'src' / '__init__.py').write_text('', encoding='utf-8')
print('LOCAL WORK ROOT =', DRIVE_ROOT)
print('COMPACT DRIVE ROOT =', PERSIST_ROOT)


In [ ]:
SUITE_SOURCE = 'from __future__ import annotations\n\nimport inspect\nimport hashlib\nimport json\nimport os\nimport platform\nimport random\nimport shutil\nimport tempfile\nimport time\nimport traceback\nfrom dataclasses import dataclass\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom datasets import Dataset, DatasetDict, load_dataset, load_from_disk\nfrom sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support\nfrom sklearn.model_selection import train_test_split\nfrom transformers import (\n    AutoModelForSequenceClassification,\n    AutoTokenizer,\n    DataCollatorWithPadding,\n    EarlyStoppingCallback,\n    Trainer,\n    TrainerCallback,\n    TrainingArguments,\n    set_seed,\n)\n\nfrom src.result_analysis import (\n    prediction_diagnostics,\n    require_matching_run_signature,\n    summarize_run_frame,\n    validate_run_frame_integrity,\n    public_run_frame,\n)\n\nROOT = Path(__file__).resolve().parents[1]\nCONFIG_PATH = ROOT / "config" / "experiment_config.json"\n_default_cache_root = ROOT / "cache"\nif "::" in str(_default_cache_root):\n    # fsspec interprets ``::`` in a local path as a chained protocol. This\n    # repository is sometimes checked out under a URL-shaped directory name.\n    cache_slug = hashlib.sha256(str(ROOT).encode("utf-8")).hexdigest()[:12]\n    _default_cache_root = Path(tempfile.gettempdir()) / f"domain_finetuning_cache_{cache_slug}"\nDATA_CACHE_ROOT = Path(os.environ.get("DOMAIN_FINETUNING_DATA_CACHE", _default_cache_root)).expanduser().resolve()\nMETHODS = ("full_ft", "lora", "adapter", "ia3", "bitfit")\n\n\ndef now_iso():\n    return datetime.now(timezone.utc).astimezone().isoformat(timespec="seconds")\n\n\ndef atomic_json(path: Path, payload):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    fd, temp_name = tempfile.mkstemp(prefix=path.name, suffix=".tmp", dir=path.parent)\n    try:\n        with os.fdopen(fd, "w", encoding="utf-8") as handle:\n            json.dump(payload, handle, ensure_ascii=False, indent=2)\n            handle.flush()\n            os.fsync(handle.fileno())\n        os.replace(temp_name, path)\n    finally:\n        if os.path.exists(temp_name):\n            os.unlink(temp_name)\n\n\ndef append_event(path: Path, payload):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("a", encoding="utf-8") as handle:\n        handle.write(json.dumps({"time": now_iso(), **payload}, ensure_ascii=False) + "\\n")\n        handle.flush()\n\n\ndef load_config():\n    return json.loads(CONFIG_PATH.read_text(encoding="utf-8"))\n\n\ndef runtime_info():\n    mps_available = bool(\n        hasattr(torch.backends, "mps") and torch.backends.mps.is_available()\n    )\n    accelerator = "cuda" if torch.cuda.is_available() else "mps" if mps_available else "cpu"\n    return {\n        "time": now_iso(),\n        "python": os.sys.version,\n        "torch": torch.__version__,\n        "cuda_runtime": torch.version.cuda,\n        "cuda_available": torch.cuda.is_available(),\n        "mps_available": mps_available,\n        "accelerator": accelerator,\n        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Apple MPS" if mps_available else "CPU",\n        "gpu_memory_gb": round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 3) if torch.cuda.is_available() else 0,\n        "machine": platform.machine(),\n    }\n\n\ndef precheck(require_cuda=True, output_path=None):\n    import datasets\n    import peft\n    import transformers\n\n    info = runtime_info() | {\n        "transformers": transformers.__version__,\n        "datasets": datasets.__version__,\n        "peft": peft.__version__,\n    }\n    if require_cuda and not torch.cuda.is_available():\n        raise RuntimeError("CUDA GPU가 감지되지 않았습니다. Python (ai_lab_first) 커널인지 확인하세요.")\n    atomic_json(Path(output_path) if output_path else ROOT / "results" / "environment.json", info)\n    return info\n\n\n@dataclass(frozen=True)\nclass TaskSpec:\n    key: str\n    path: str\n    subset: str | None\n    text_col: str\n    label_col: str\n    num_labels: int\n    source_split: str | None = None\n    label_threshold: float | None = None\n    direct: str | None = None\n\n\nTASKS = {\n    "measuring_hate_speech": TaskSpec("measuring_hate_speech", "ucberkeley-dlab/measuring-hate-speech", None, "comment", "hatespeech", 2, "train", 1.0),\n    "tweet_sentiment": TaskSpec("tweet_sentiment", "cardiffnlp/tweet_eval", "sentiment", "text", "label", 3),\n    "finance_sentiment": TaskSpec("finance_sentiment", "lmassaron/FinancialPhraseBank", None, "sentence", "label", 3),\n    "movie_reviews": TaskSpec("movie_reviews", "stanfordnlp/imdb", None, "text", "label", 2),\n    "product_reviews": TaskSpec("product_reviews", "SetFit/amazon_reviews_multi_en", None, "text", "label", 5),\n    "tweet_emotion": TaskSpec("tweet_emotion", "cardiffnlp/tweet_eval", "emotion", "text", "label", 4),\n    "tweet_hate": TaskSpec("tweet_hate", "cardiffnlp/tweet_eval", "hate", "text", "label", 2),\n    "tweet_offensive": TaskSpec("tweet_offensive", "cardiffnlp/tweet_eval", "offensive", "text", "label", 2),\n    "tweet_irony": TaskSpec("tweet_irony", "cardiffnlp/tweet_eval", "irony", "text", "label", 2),\n    "news_topic": TaskSpec("news_topic", "fancyzhx/ag_news", None, "text", "label", 4),\n    "news_ynat": TaskSpec("news_ynat", "klue", "ynat", "title", "label", 7),\n    "movie_nsmc": TaskSpec("movie_nsmc", "csv", None, "document", "label", 2, direct="nsmc"),\n    "comment_kmhas_binary": TaskSpec("comment_kmhas_binary", "csv", None, "text", "label", 2, direct="kmhas"),\n}\n\n\ndef _raw_dataset(spec: TaskSpec):\n    if spec.direct == "nsmc":\n        return load_dataset("csv", data_files={\n            "train": "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt",\n            "test": "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt",\n        }, delimiter="\\t")\n    if spec.direct == "kmhas":\n        return load_dataset("csv", data_files={\n            "train": "https://raw.githubusercontent.com/adlnlp/K-MHaS/main/data/kmhas_train.txt",\n            "validation": "https://raw.githubusercontent.com/adlnlp/K-MHaS/main/data/kmhas_valid.txt",\n            "test": "https://raw.githubusercontent.com/adlnlp/K-MHaS/main/data/kmhas_test.txt",\n        }, delimiter="\\t", column_names=["text", "label"], skiprows=1)\n    return load_dataset(spec.path, spec.subset) if spec.subset else load_dataset(spec.path)\n\n\ndef _kmhas_binary(value):\n    if hasattr(value, "tolist"):\n        value = value.tolist()\n    if isinstance(value, str):\n        try:\n            value = json.loads(value)\n        except Exception:\n            value = [int(x.strip()) for x in value.split(",") if x.strip()]\n    if isinstance(value, (int, np.integer)):\n        value = [int(value)]\n    return 0 if list(value) == [8] else 1\n\n\ndef _standardize(raw, spec: TaskSpec):\n    if spec.source_split:\n        raw = DatasetDict(all=raw[spec.source_split])\n    standardized = {}\n    for split_name, split in raw.items():\n        texts, labels, ids = [], [], []\n        columns = set(split.column_names)\n        text_col = spec.text_col if spec.text_col in columns else next((x for x in ("text", "sentence", "comment", "document", "title") if x in columns), None)\n        label_col = spec.label_col if spec.label_col in columns else next((x for x in ("label", "labels", "hatespeech", "hate_speech_score") if x in columns), None)\n        if text_col is None or label_col is None:\n            raise RuntimeError(f"{spec.key}: text/label 컬럼 확인 실패: {split.column_names}")\n        for i, row in enumerate(split):\n            text = row.get(text_col)\n            raw_label = row.get(label_col)\n            if text is None or raw_label is None:\n                continue\n            if spec.key == "comment_kmhas_binary":\n                label = _kmhas_binary(raw_label)\n            elif spec.label_threshold is not None:\n                try:\n                    label = int(float(raw_label) >= spec.label_threshold)\n                except (TypeError, ValueError):\n                    continue\n            else:\n                label = int(raw_label)\n            if not 0 <= label < spec.num_labels:\n                continue\n            texts.append(str(text)); labels.append(label); ids.append(f"{spec.key}:{split_name}:{i}")\n        standardized[split_name] = Dataset.from_dict({"sample_id": ids, "text": texts, "labels": labels})\n    return DatasetDict(standardized)\n\n\ndef _split_indices(labels, test_size, seed):\n    idx = np.arange(len(labels))\n    try:\n        return train_test_split(idx, test_size=test_size, random_state=seed, stratify=np.asarray(labels))\n    except ValueError:\n        return train_test_split(idx, test_size=test_size, random_state=seed)\n\n\ndef _ensure_three_splits(ds: DatasetDict, seed=42):\n    if "all" in ds:\n        train_idx, hold_idx = _split_indices(ds["all"]["labels"], 0.2, seed)\n        hold = ds["all"].select(sorted(hold_idx.tolist()))\n        val_idx, test_idx = _split_indices(hold["labels"], 0.5, seed)\n        return DatasetDict(train=ds["all"].select(sorted(train_idx.tolist())), validation=hold.select(sorted(val_idx.tolist())), test=hold.select(sorted(test_idx.tolist())))\n    if all(x in ds for x in ("train", "validation", "test")):\n        return DatasetDict({x: ds[x] for x in ("train", "validation", "test")})\n    if "test" in ds:\n        train_idx, val_idx = _split_indices(ds["train"]["labels"], 0.1, seed)\n        return DatasetDict(train=ds["train"].select(sorted(train_idx.tolist())), validation=ds["train"].select(sorted(val_idx.tolist())), test=ds["test"])\n    if "validation" in ds:\n        train_idx, val_idx = _split_indices(ds["train"]["labels"], 0.1, seed)\n        return DatasetDict(train=ds["train"].select(sorted(train_idx.tolist())), validation=ds["train"].select(sorted(val_idx.tolist())), test=ds["validation"])\n    train_idx, hold_idx = _split_indices(ds["train"]["labels"], 0.2, seed)\n    hold = ds["train"].select(sorted(hold_idx.tolist()))\n    val_idx, test_idx = _split_indices(hold["labels"], 0.5, seed)\n    return DatasetDict(train=ds["train"].select(sorted(train_idx.tolist())), validation=hold.select(sorted(val_idx.tolist())), test=hold.select(sorted(test_idx.tolist())))\n\n\ndef _balanced_limit(ds: Dataset, limit: int | None, seed=42):\n    if not limit or len(ds) <= limit:\n        return ds\n    labels = np.asarray(ds["labels"]); idx = np.arange(len(ds))\n    chosen, _ = train_test_split(idx, train_size=limit, random_state=seed, stratify=labels)\n    return ds.select(sorted(chosen.tolist()))\n\n\ndef load_task(task_key, run_mode, limits=None):\n    if run_mode == "SMOKE":\n        cache_tag = "smoke"\n    elif limits:\n        cache_tag = "paper_" + "_".join(f"{k}{int(v)}" for k, v in sorted(limits.items()))\n    else:\n        cache_tag = "paper_full"\n    cache = DATA_CACHE_ROOT / task_key / cache_tag\n    cache_complete = (cache / "dataset_dict.json").exists() and all(\n        (cache / split / "state.json").exists() and (cache / split / "dataset_info.json").exists()\n        for split in ("train", "validation", "test")\n    )\n    if cache_complete:\n        return load_from_disk(str(cache))\n    if cache.exists():\n        broken = cache.with_name(cache.name + ".incomplete_" + datetime.now().strftime("%Y%m%d_%H%M%S"))\n        cache.rename(broken)\n    spec = TASKS[task_key]\n    ds = _ensure_three_splits(_standardize(_raw_dataset(spec), spec))\n    cfg = load_config()\n    if run_mode == "SMOKE":\n        limits = cfg["smoke_limits"]\n    if limits:\n        ds = DatasetDict({name: _balanced_limit(split, int(limits[name]), 42) for name, split in ds.items()})\n    cache.parent.mkdir(parents=True, exist_ok=True)\n    temp_cache = cache.with_name(cache.name + ".building")\n    if temp_cache.exists():\n        shutil.rmtree(temp_cache)\n    ds.save_to_disk(str(temp_cache))\n    temp_cache.rename(cache)\n    atomic_json(cache.parent / f"{cache_tag}_manifest.json", {\n        "task": task_key, "run_mode": run_mode, "source": spec.path, "subset": spec.subset,\n        "limits": limits, "rows": {k: len(v) for k, v in ds.items()}, "split_seed": 42,\n        "fingerprints": {k: getattr(v, "_fingerprint", "UNKNOWN") for k, v in ds.items()},\n    })\n    return ds\n\n\nclass BottleneckAdapter(nn.Module):\n    def __init__(self, hidden_size, bottleneck, dropout=0.0):\n        super().__init__()\n        self.down = nn.Linear(hidden_size, bottleneck)\n        self.activation = nn.GELU()\n        self.dropout = nn.Dropout(dropout)\n        self.up = nn.Linear(bottleneck, hidden_size)\n        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)\n\n    def forward(self, hidden_states):\n        return hidden_states + self.up(self.dropout(self.activation(self.down(hidden_states))))\n\n\nclass OutputWithAdapter(nn.Module):\n    def __init__(self, original, hidden_size, bottleneck, dropout):\n        super().__init__()\n        self.dense = original.dense\n        self.LayerNorm = original.LayerNorm\n        self.dropout = original.dropout\n        self.adapter = BottleneckAdapter(hidden_size, bottleneck, dropout)\n\n    def forward(self, hidden_states, input_tensor):\n        hidden_states = self.dense(hidden_states)\n        hidden_states = self.adapter(hidden_states)\n        hidden_states = self.dropout(hidden_states)\n        return self.LayerNorm(hidden_states + input_tensor)\n\n\ndef _encoder_layers(model):\n    for attr in ("roberta", "bert", "deberta", "electra"):\n        base = getattr(model, attr, None)\n        if base is not None and hasattr(base, "encoder") and hasattr(base.encoder, "layer"):\n            return base.encoder.layer\n    raise RuntimeError(f"Adapter 미지원 모델 구조: {model.__class__.__name__}")\n\n\ndef _unfreeze_head(model):\n    for name, param in model.named_parameters():\n        if any(token in name for token in ("classifier", "score")):\n            param.requires_grad = True\n\n\ndef build_model(model_name, num_labels, method, cfg, revision=None):\n    pretrained_kwargs = {"revision": revision} if revision else {}\n    model = AutoModelForSequenceClassification.from_pretrained(\n        model_name,\n        num_labels=num_labels,\n        ignore_mismatched_sizes=True,\n        attn_implementation="eager",\n        **pretrained_kwargs,\n    )\n    if method == "full_ft":\n        return model\n    if method == "bitfit":\n        for param in model.parameters(): param.requires_grad = False\n        for name, param in model.named_parameters():\n            if name.endswith(".bias"): param.requires_grad = True\n        _unfreeze_head(model)\n        return model\n    if method == "adapter":\n        for param in model.parameters(): param.requires_grad = False\n        acfg = cfg["adapter"]\n        for layer in _encoder_layers(model):\n            layer.output = OutputWithAdapter(layer.output, model.config.hidden_size, acfg["bottleneck"], acfg["dropout"])\n        _unfreeze_head(model)\n        return model\n    from peft import IA3Config, LoraConfig, TaskType, get_peft_model\n    if method == "lora":\n        lcfg = cfg["lora"]\n        peft_cfg = LoraConfig(task_type=TaskType.SEQ_CLS, r=lcfg["r"], lora_alpha=lcfg["alpha"], lora_dropout=lcfg["dropout"], target_modules=["query", "value"], modules_to_save=["classifier"], bias="none")\n    elif method == "ia3":\n        peft_cfg = IA3Config(task_type=TaskType.SEQ_CLS, target_modules=["key", "value", "intermediate.dense"], feedforward_modules=["intermediate.dense"], modules_to_save=["classifier"])\n    else:\n        raise ValueError(method)\n    return get_peft_model(model, peft_cfg)\n\n\ndef parameter_counts(model):\n    total = sum(p.numel() for p in model.parameters())\n    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)\n    return {"trainable_params": trainable, "total_params": total, "trainable_parameter_ratio": trainable / total}\n\n\ndef compute_metrics(prediction):\n    labels = prediction.label_ids\n    preds = np.argmax(prediction.predictions, axis=-1)\n    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)\n    return {"accuracy": accuracy_score(labels, preds), "macro_f1": f1, "macro_precision": precision, "macro_recall": recall}\n\n\nclass AtomicEpochCallback(TrainerCallback):\n    def __init__(self, path):\n        self.path = Path(path)\n        if self.path.exists():\n            try:\n                self.rows = pd.read_csv(self.path).to_dict("records")\n            except Exception:\n                self.rows = []\n        else:\n            self.rows = []\n\n    def _save(self):\n        self.path.parent.mkdir(parents=True, exist_ok=True)\n        temp = self.path.with_suffix(".tmp")\n        pd.DataFrame(self.rows).to_csv(temp, index=False, encoding="utf-8-sig")\n        os.replace(temp, self.path)\n\n    def on_log(self, args, state, control, logs=None, **kwargs):\n        if logs:\n            self.rows.append({"time": now_iso(), "event": "log", "epoch": state.epoch, "global_step": state.global_step, **logs})\n            self._save()\n\n    def on_save(self, args, state, control, **kwargs):\n        self.rows.append({"time": now_iso(), "event": "checkpoint", "epoch": state.epoch, "global_step": state.global_step})\n        self._save()\n\n\ndef training_args(run_dir, method, seed, epochs, run_mode, cfg):\n    return _training_args(run_dir, method, seed, epochs, run_mode, cfg, overrides=None)\n\n\ndef _training_args(run_dir, method, seed, epochs, run_mode, cfg, overrides=None):\n    overrides = overrides or {}\n    requested_precision = overrides.get("precision", cfg["precision"])\n    use_fp16 = requested_precision == "fp16" and torch.cuda.is_available()\n    kwargs = dict(\n        output_dir=str(run_dir / "checkpoints"),\n        learning_rate=overrides.get("learning_rate", cfg["learning_rates"][method]),\n        per_device_train_batch_size=cfg["batch_size"],\n        per_device_eval_batch_size=cfg["eval_batch_size"],\n        gradient_accumulation_steps=cfg["gradient_accumulation_steps"],\n        num_train_epochs=1 if run_mode == "SMOKE" else overrides.get("epochs", epochs),\n        weight_decay=cfg["weight_decay"], warmup_ratio=cfg["warmup_ratio"],\n        logging_strategy="steps", logging_steps=20,\n        save_strategy="epoch", load_best_model_at_end=True,\n        metric_for_best_model="macro_f1", greater_is_better=True,\n        save_total_limit=1, report_to="none",\n        seed=seed, data_seed=42, fp16=use_fp16,\n        dataloader_num_workers=cfg["dataloader_num_workers"],\n        optim=cfg.get("optimizer", "adamw_torch"),\n        lr_scheduler_type=cfg.get("lr_scheduler_type", "linear"),\n    )\n    signature = inspect.signature(TrainingArguments.__init__)\n    kwargs["eval_strategy" if "eval_strategy" in signature.parameters else "evaluation_strategy"] = "epoch"\n    return TrainingArguments(**kwargs)\n\n\ndef _class_weights(labels, num_labels, strategy):\n    if strategy in (None, "none"):\n        return None\n    counts = np.bincount(np.asarray(labels, dtype=int), minlength=num_labels).astype(float)\n    if np.any(counts == 0):\n        raise ValueError(f"class weighting requires every class in train split; counts={counts.tolist()}")\n    if strategy == "inverse_frequency":\n        weights = counts.sum() / (num_labels * counts)\n    elif strategy == "inverse_sqrt_frequency":\n        weights = 1.0 / np.sqrt(counts)\n        weights /= weights.mean()\n    else:\n        raise ValueError(f"unknown class-weighting strategy: {strategy}")\n    return weights.astype(np.float32)\n\n\nclass WeightedTrainer(Trainer):\n    def __init__(self, *args, class_weights, **kwargs):\n        super().__init__(*args, **kwargs)\n        self.class_weights = torch.as_tensor(class_weights, dtype=torch.float32)\n\n    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):\n        labels = inputs.get("labels")\n        outputs = model(**inputs)\n        logits = outputs.get("logits") if isinstance(outputs, dict) else outputs.logits\n        loss = F.cross_entropy(\n            logits.reshape(-1, logits.shape[-1]),\n            labels.reshape(-1),\n            weight=self.class_weights.to(logits.device),\n        )\n        return (loss, outputs) if return_outputs else loss\n\n\ndef _latest_checkpoint(run_dir):\n    root = run_dir / "checkpoints"\n    candidates = sorted(\n        (p for p in root.glob("checkpoint-*") if (p / "trainer_state.json").exists()),\n        key=lambda p: int(p.name.split("-")[-1]),\n    ) if root.exists() else []\n    return str(candidates[-1]) if candidates else None\n\n\ndef run_one(\n    study,\n    task_key,\n    model_name,\n    method,\n    seed,\n    run_mode,\n    epochs,\n    limits=None,\n    *,\n    experiment_id=None,\n    variant=None,\n    training_overrides=None,\n    class_weighting="none",\n    keep_checkpoint=None,\n    force=False,\n):\n    cfg = load_config()\n    if method not in METHODS: raise ValueError(method)\n    model_slug = model_name.replace("/", "__")\n    if experiment_id:\n        run_dir = ROOT / "results" / "followup" / experiment_id / (variant or "default") / task_key / model_slug / method / f"seed_{seed}"\n    else:\n        run_dir = ROOT / "results" / study / run_mode / task_key / model_slug / method / f"seed_{seed}"\n    signature_payload = {\n        "study": study,\n        "task": task_key,\n        "model": model_name,\n        "method": method,\n        "seed": seed,\n        "run_mode": run_mode,\n        "epochs": epochs,\n        "limits": limits,\n        "experiment_id": experiment_id,\n        "variant": variant,\n        "training_overrides": training_overrides or {},\n        "class_weighting": class_weighting,\n        "keep_checkpoint": keep_checkpoint,\n        "protocol": cfg,\n        "suite_sha256": hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),\n    }\n    run_signature = hashlib.sha256(\n        json.dumps(signature_payload, sort_keys=True, ensure_ascii=False).encode("utf-8")\n    ).hexdigest()\n    metrics_path = run_dir / "final_metrics.json"\n    if metrics_path.exists():\n        try:\n            existing = json.loads(metrics_path.read_text(encoding="utf-8"))\n            if existing.get("status") == "COMPLETE" and not force:\n                require_matching_run_signature(existing, run_signature, metrics_path)\n                return existing\n        except (json.JSONDecodeError, OSError):\n            pass\n    if force and run_dir.exists():\n        archived = run_dir.with_name(run_dir.name + ".superseded_" + datetime.now().strftime("%Y%m%d_%H%M%S"))\n        run_dir.rename(archived)\n    run_dir.mkdir(parents=True, exist_ok=True)\n    checkpoint_at_start = _latest_checkpoint(run_dir)\n    previous_status = {}\n    if (run_dir / "status.json").exists():\n        try: previous_status = json.loads((run_dir / "status.json").read_text(encoding="utf-8"))\n        except Exception: previous_status = {}\n    if checkpoint_at_start:\n        require_matching_run_signature(\n            previous_status,\n            run_signature,\n            run_dir / "status.json",\n        )\n    status = {\n        "status": "RUNNING", "started_at": previous_status.get("started_at", now_iso()),\n        "resumed_at": now_iso() if checkpoint_at_start else None,\n        "resume_count": int(previous_status.get("resume_count", 0)) + (1 if checkpoint_at_start else 0),\n        "resumed_from_checkpoint": checkpoint_at_start,\n        "study": study, "task": task_key, "model": model_name, "method": method,\n        "seed": seed, "run_mode": run_mode, "experiment_id": experiment_id,\n        "variant": variant, "run_signature": run_signature,\n    }\n    atomic_json(run_dir / "status.json", status)\n    append_event(run_dir / "events.jsonl", {"event": "RUN_STARTED", **status})\n    try:\n        set_seed(seed)\n        ds = load_task(task_key, run_mode, limits)\n        spec = TASKS[task_key]\n        revision = cfg.get("model_revisions", {}).get(model_name)\n        pretrained_kwargs = {"revision": revision} if revision else {}\n        tokenizer = AutoTokenizer.from_pretrained(\n            model_name,\n            use_fast=False if "bertweet" in model_name.lower() else True,\n            **pretrained_kwargs,\n        )\n        def tokenize(batch): return tokenizer(batch["text"], truncation=True, max_length=cfg["max_length"])\n        tokenized = ds.map(tokenize, batched=True, remove_columns=["sample_id", "text"])\n        model = build_model(model_name, spec.num_labels, method, cfg, revision=revision)\n        counts = parameter_counts(model)\n        class_weights = _class_weights(ds["train"]["labels"], spec.num_labels, class_weighting)\n        atomic_json(run_dir / "run_config.json", {\n            **status, "epochs": epochs, "hyperparameters": cfg, "runtime": runtime_info(),\n            "model_commit": getattr(model.config, "_commit_hash", None), **counts,\n            "training_overrides": training_overrides or {}, "class_weighting": class_weighting,\n            "class_weights": class_weights.tolist() if class_weights is not None else None,\n            "train_label_counts": np.bincount(np.asarray(ds["train"]["labels"]), minlength=spec.num_labels).astype(int).tolist(),\n        })\n        callback = AtomicEpochCallback(run_dir / "epoch_metrics.csv")\n        trainer_kwargs = dict(\n            model=model, args=_training_args(run_dir, method, seed, epochs, run_mode, cfg, training_overrides),\n            train_dataset=tokenized["train"], eval_dataset=tokenized["validation"],\n            data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics,\n            callbacks=[EarlyStoppingCallback(early_stopping_patience=cfg["early_stopping_patience"]), callback],\n        )\n        if "processing_class" in inspect.signature(Trainer.__init__).parameters: trainer_kwargs["processing_class"] = tokenizer\n        else: trainer_kwargs["tokenizer"] = tokenizer\n        trainer = WeightedTrainer(class_weights=class_weights, **trainer_kwargs) if class_weights is not None else Trainer(**trainer_kwargs)\n        trainer_device = str(trainer.args.device)\n        checkpoint = _latest_checkpoint(run_dir)\n        started = time.perf_counter()\n        trainer.train(resume_from_checkpoint=checkpoint)\n        train_seconds = time.perf_counter() - started\n        test = trainer.predict(tokenized["test"], metric_key_prefix="test")\n        predictions = np.argmax(test.predictions, axis=-1)\n        prediction_profile = prediction_diagnostics(test.label_ids, predictions, spec.num_labels)\n        pd.DataFrame({"sample_id": ds["test"]["sample_id"], "label": test.label_ids, "prediction": predictions}).to_csv(run_dir / "predictions.csv", index=False, encoding="utf-8-sig")\n        history = pd.DataFrame(trainer.state.log_history)\n        history.to_csv(run_dir / "trainer_history.csv", index=False, encoding="utf-8-sig")\n        result = {\n            "status": "COMPLETE", "completed_at": now_iso(), "study": study, "task": task_key,\n            "model": model_name, "method": method, "seed": seed, "run_mode": run_mode,\n            "experiment_id": experiment_id, "variant": variant, "run_signature": run_signature,\n            "train_rows": len(ds["train"]), "validation_rows": len(ds["validation"]), "test_rows": len(ds["test"]),\n            "epochs_requested": 1 if run_mode == "SMOKE" else (training_overrides or {}).get("epochs", epochs),\n            "epochs_completed": float(trainer.state.epoch or 0.0),\n            "global_step": int(trainer.state.global_step),\n            "best_validation_macro_f1": float(trainer.state.best_metric) if trainer.state.best_metric is not None else None,\n            "learning_rate": (training_overrides or {}).get("learning_rate", cfg["learning_rates"][method]),\n            "class_weighting": class_weighting,\n            "trainer_device": trainer_device,\n            "train_seconds": train_seconds, **counts, **test.metrics, "runtime": runtime_info(),\n            "best_checkpoint": (\n                Path(trainer.state.best_model_checkpoint).name\n                if trainer.state.best_model_checkpoint else None\n            ),\n            **prediction_profile,\n        }\n        atomic_json(metrics_path, result)\n        atomic_json(run_dir / "status.json", result)\n        append_event(run_dir / "events.jsonl", {\n            "event": "RUN_COMPLETE", "test_macro_f1": result.get("test_macro_f1"),\n            "constant_prediction_collapse": result["constant_prediction_collapse"],\n        })\n        should_keep_checkpoint = cfg["keep_best_checkpoint"] if keep_checkpoint is None else keep_checkpoint\n        if not should_keep_checkpoint:\n            shutil.rmtree(run_dir / "checkpoints", ignore_errors=True)\n        del trainer, model\n        if torch.cuda.is_available(): torch.cuda.empty_cache()\n        return result\n    except Exception as exc:\n        failed = {**status, "status": "FAILED", "failed_at": now_iso(), "error_type": type(exc).__name__, "error": str(exc)}\n        atomic_json(run_dir / "status.json", failed)\n        (run_dir / "error.txt").write_text(traceback.format_exc(), encoding="utf-8")\n        append_event(run_dir / "events.jsonl", {"event": "RUN_FAILED", "error": str(exc)})\n        if torch.cuda.is_available(): torch.cuda.empty_cache()\n        raise\n\n\ndef build_jobs(study):\n    cfg = load_config(); section = cfg[study]\n    mode_limits = section.get("limits")\n    jobs = []\n    for task in section["tasks"]:\n        for model in section["models"]:\n            for method in cfg["methods"]:\n                for seed in cfg["seeds"]:\n                    jobs.append({"study": study, "task_key": task, "model_name": model, "method": method, "seed": seed, "epochs": section["epochs"], "limits": mode_limits})\n    return jobs\n\n\ndef run_study(study, run_mode="SMOKE", max_jobs=None, continue_on_error=None):\n    if run_mode not in {"SMOKE", "PAPER"}: raise ValueError("run_mode은 SMOKE 또는 PAPER만 가능합니다.")\n    precheck(require_cuda=True)\n    cfg = load_config(); jobs = build_jobs(study)\n    if max_jobs is not None: jobs = jobs[:max_jobs]\n    continue_on_error = cfg["continue_on_error"] if continue_on_error is None else continue_on_error\n    progress_path = ROOT / "results" / study / run_mode / "progress.json"\n    events_path = ROOT / "results" / study / run_mode / "events.jsonl"\n    completed, failed = 0, 0\n    results = []\n    for index, job in enumerate(jobs, 1):\n        current = {"study": study, "run_mode": run_mode, "total": len(jobs), "index": index, "completed": completed, "failed": failed, "current": job, "updated_at": now_iso()}\n        atomic_json(progress_path, current); append_event(events_path, {"event": "JOB_DISPATCH", "index": index, **job})\n        try:\n            result = run_one(run_mode=run_mode, **job)\n            results.append(result); completed += 1\n        except Exception:\n            failed += 1\n            if not continue_on_error:\n                atomic_json(progress_path, {**current, "status": "STOPPED_ON_ERROR", "failed": failed, "updated_at": now_iso()})\n                raise\n        atomic_json(progress_path, {**current, "status": "RUNNING", "completed": completed, "failed": failed, "updated_at": now_iso()})\n    final = {"study": study, "run_mode": run_mode, "status": "COMPLETE" if failed == 0 else "COMPLETE_WITH_ERRORS", "total": len(jobs), "completed": completed, "failed": failed, "updated_at": now_iso()}\n    atomic_json(progress_path, final); append_event(events_path, {"event": "STUDY_FINISHED", **final})\n    return pd.DataFrame(results)\n\n\ndef aggregate(run_mode="PAPER", *, strict=True):\n    """Aggregate broad-benchmark runs after provenance and seed checks.\n\n    ``strict=False`` permits an intentionally partial progress export. Final\n    PAPER tables should always use the default strict validation.\n    """\n    if run_mode not in {"SMOKE", "PAPER"}:\n        raise ValueError("run_mode은 SMOKE 또는 PAPER만 가능합니다.")\n    cfg = load_config()\n    rows = []\n    for path in (ROOT / "results").glob(f"study*/{run_mode}/**/final_metrics.json"):\n        row = json.loads(path.read_text(encoding="utf-8"))\n        relative_parts = path.relative_to(ROOT / "results").parts\n        if len(relative_parts) != 7:\n            raise ValueError(f"unexpected result path layout: {path}")\n        path_study, path_mode, path_task, path_model, path_method, path_seed, _ = relative_parts\n        expected_path_identity = (\n            str(row.get("study")),\n            str(row.get("run_mode")),\n            str(row.get("task")),\n            str(row.get("model", "")).replace("/", "__"),\n            str(row.get("method")),\n            f"seed_{row.get(\'seed\')}",\n        )\n        if expected_path_identity != (\n            path_study,\n            path_mode,\n            path_task,\n            path_model,\n            path_method,\n            path_seed,\n        ):\n            raise ValueError(f"result payload/path identity mismatch: {path}")\n        row["source_file"] = str(path.relative_to(ROOT))\n        rows.append(row)\n    frame = pd.DataFrame(rows)\n    expected_keys = None\n    if strict:\n        expected_keys = {\n            (study, task, model, method, int(seed))\n            for study in ("study1", "study2", "study3")\n            for task in cfg[study]["tasks"]\n            for model in cfg[study]["models"]\n            for method in cfg["methods"]\n            for seed in cfg["seeds"]\n        }\n    validate_run_frame_integrity(\n        frame,\n        expected_seeds=cfg["seeds"] if strict else None,\n        expected_run_mode=run_mode,\n        expected_keys=expected_keys,\n    )\n    summary = summarize_run_frame(\n        frame,\n        expected_seeds=cfg["seeds"] if strict else None,\n    )\n    if strict and not bool(summary["seed_coverage_ok"].all()):\n        raise ValueError("aggregate contains a group with invalid seed coverage")\n    out = ROOT / "results" / "aggregate"; out.mkdir(parents=True, exist_ok=True)\n    public_run_frame(frame).to_csv(\n        out / f"all_runs_{run_mode.lower()}.csv",\n        index=False,\n        encoding="utf-8-sig",\n        float_format="%.17g",\n    )\n    summary.to_csv(\n        out / f"summary_{run_mode.lower()}.csv",\n        index=False,\n        encoding="utf-8-sig",\n        float_format="%.17g",\n    )\n    return frame\n'
(DRIVE_ROOT / 'src' / 'suite.py').write_text(SUITE_SOURCE, encoding='utf-8')
print('suite.py written:', len(SUITE_SOURCE), 'chars')


In [ ]:
RESULT_ANALYSIS_SOURCE = 'from __future__ import annotations\n\nimport itertools\nimport json\nimport math\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\n\nMETHOD_LABELS = {\n    "full_ft": "Full Fine-tuning",\n    "lora": "LoRA",\n    "adapter": "Adapter",\n    "ia3": "IA3",\n    "bitfit": "BitFit",\n}\n\nTASK_NUM_LABELS = {\n    "measuring_hate_speech": 2,\n    "tweet_sentiment": 3,\n    "finance_sentiment": 3,\n    "movie_reviews": 2,\n    "product_reviews": 5,\n    "tweet_emotion": 4,\n    "tweet_hate": 2,\n    "tweet_offensive": 2,\n    "tweet_irony": 2,\n    "news_topic": 4,\n    "news_ynat": 7,\n    "movie_nsmc": 2,\n    "comment_kmhas_binary": 2,\n}\n\nT_CRITICAL_975 = {4: 2.7764451051977987}\n\n# Compact, publication-safe run fields.  Per-example predictions, local paths,\n# checkpoint names, timestamps, and machine-specific runtime dictionaries stay\n# in the ignored per-run artifacts rather than public aggregate CSVs.\nPUBLIC_RUN_COLUMNS = (\n    "status",\n    "study",\n    "task",\n    "model",\n    "method",\n    "seed",\n    "run_mode",\n    "experiment_id",\n    "variant",\n    "run_signature",\n    "train_rows",\n    "validation_rows",\n    "test_rows",\n    "epochs_requested",\n    "learning_rate",\n    "class_weighting",\n    "trainer_device",\n    "train_seconds",\n    "trainable_params",\n    "total_params",\n    "trainable_parameter_ratio",\n    "test_loss",\n    "test_accuracy",\n    "test_macro_f1",\n    "test_macro_precision",\n    "test_macro_recall",\n    "true_label_counts",\n    "predicted_label_counts",\n    "predicted_class_count",\n    "predicted_class_coverage",\n    "majority_prediction_rate",\n    "normalized_prediction_entropy",\n    "constant_prediction_collapse",\n    "near_constant_prediction",\n    "epochs_completed",\n    "global_step",\n    "best_validation_macro_f1",\n)\n\n\ndef public_run_frame(frame: pd.DataFrame) -> pd.DataFrame:\n    """Return compact run metrics suitable for the public release surface."""\n    columns = [name for name in PUBLIC_RUN_COLUMNS if name in frame.columns]\n    public = frame.loc[:, columns].copy()\n    identity = [\n        name\n        for name in ("experiment_id", "variant", "study", "task", "model", "method", "seed")\n        if name in public.columns\n    ]\n    return public.sort_values(identity, kind="stable").reset_index(drop=True)\n\n\ndef require_matching_run_signature(\n    payload: dict,\n    current_signature: str,\n    artifact_path: Path | str,\n) -> None:\n    """Reject cached or resumable artifacts from a different run definition."""\n    existing_signature = payload.get("run_signature")\n    if existing_signature != current_signature:\n        raise RuntimeError(\n            f"stale run artifact at {artifact_path}; "\n            "the saved run signature does not match the current code/configuration. "\n            "Use force=True or a new experiment_id."\n        )\n\n\ndef validate_run_frame_integrity(\n    frame: pd.DataFrame,\n    *,\n    expected_seeds: list[int] | None,\n    expected_run_mode: str,\n    expected_keys: set[tuple[str, str, str, str, int]] | None = None,\n) -> None:\n    """Fail before aggregation when status, identity, or seed coverage is unsafe."""\n    identity_columns = ["study", "task", "model", "method", "seed"]\n    required = set(identity_columns) | {"status", "run_mode"}\n    missing = required - set(frame.columns)\n    if missing:\n        raise ValueError(f"run table is missing integrity columns: {sorted(missing)}")\n    if frame.empty:\n        raise ValueError("run table is empty")\n\n    incomplete = frame[frame["status"] != "COMPLETE"]\n    if not incomplete.empty:\n        raise ValueError(f"run table contains {len(incomplete)} non-COMPLETE rows")\n\n    wrong_mode = frame[frame["run_mode"] != expected_run_mode]\n    if not wrong_mode.empty:\n        found = sorted(set(str(value) for value in wrong_mode["run_mode"]))\n        raise ValueError(\n            f"run table contains modes other than {expected_run_mode}: {found}"\n        )\n\n    duplicate_mask = frame.duplicated(identity_columns, keep=False)\n    if duplicate_mask.any():\n        duplicate_count = int(duplicate_mask.sum())\n        raise ValueError(f"run table contains {duplicate_count} rows with duplicate run keys")\n\n    if expected_seeds is not None:\n        expected_seed_set = sorted(int(seed) for seed in expected_seeds)\n        bad_groups = []\n        for keys, group in frame.groupby(identity_columns[:-1], sort=True, dropna=False):\n            seeds = sorted(int(seed) for seed in group["seed"])\n            if seeds != expected_seed_set:\n                bad_groups.append((keys, seeds))\n        if bad_groups:\n            preview = "; ".join(f"{keys}: {seeds}" for keys, seeds in bad_groups[:3])\n            raise ValueError(\n                f"run table has {len(bad_groups)} groups with incomplete seed coverage; {preview}"\n            )\n\n    if expected_keys is not None:\n        actual_keys = {\n            (\n                str(row.study),\n                str(row.task),\n                str(row.model),\n                str(row.method),\n                int(row.seed),\n            )\n            for row in frame[identity_columns].itertuples(index=False)\n        }\n        missing_keys = expected_keys - actual_keys\n        unexpected_keys = actual_keys - expected_keys\n        if missing_keys or unexpected_keys:\n            raise ValueError(\n                "run table does not match the configured design: "\n                f"missing={len(missing_keys)}, unexpected={len(unexpected_keys)}"\n            )\n\n\ndef constant_prediction_expected(accuracy: float, num_labels: int) -> dict[str, float]:\n    """Metrics produced when every sample is assigned to one class.\n\n    If the predicted class has prevalence ``accuracy``, macro precision is\n    accuracy/K, macro recall is 1/K, and macro F1 is\n    2*accuracy/(1+accuracy)/K. This fingerprint can be checked from an\n    aggregate table even when per-example predictions are unavailable.\n    """\n    return {\n        "precision": accuracy / num_labels,\n        "recall": 1.0 / num_labels,\n        "f1": (2.0 * accuracy / (1.0 + accuracy)) / num_labels,\n    }\n\n\ndef has_constant_prediction_fingerprint(\n    *,\n    accuracy: float,\n    precision: float,\n    recall: float,\n    f1: float,\n    num_labels: int,\n    atol: float = 1e-12,\n) -> bool:\n    expected = constant_prediction_expected(accuracy, num_labels)\n    observed = {"precision": precision, "recall": recall, "f1": f1}\n    return all(math.isclose(observed[key], expected[key], rel_tol=0.0, abs_tol=atol) for key in expected)\n\n\ndef prediction_diagnostics(labels, predictions, num_labels: int) -> dict:\n    labels = np.asarray(labels, dtype=int)\n    predictions = np.asarray(predictions, dtype=int)\n    true_counts = np.bincount(labels, minlength=num_labels)\n    predicted_counts = np.bincount(predictions, minlength=num_labels)\n    total = int(predicted_counts.sum())\n    nonzero = predicted_counts[predicted_counts > 0]\n    probabilities = nonzero / total if total else np.asarray([], dtype=float)\n    entropy = float(-(probabilities * np.log(probabilities)).sum()) if probabilities.size else 0.0\n    normalized_entropy = entropy / math.log(num_labels) if num_labels > 1 else 0.0\n    predicted_class_count = int(np.count_nonzero(predicted_counts))\n    majority_prediction_rate = float(predicted_counts.max() / total) if total else 0.0\n    return {\n        "true_label_counts": true_counts.astype(int).tolist(),\n        "predicted_label_counts": predicted_counts.astype(int).tolist(),\n        "predicted_class_count": predicted_class_count,\n        "predicted_class_coverage": predicted_class_count / num_labels,\n        "majority_prediction_rate": majority_prediction_rate,\n        "normalized_prediction_entropy": normalized_entropy,\n        "constant_prediction_collapse": predicted_class_count == 1,\n        "near_constant_prediction": majority_prediction_rate >= 0.98,\n    }\n\n\ndef paired_difference_stats(baseline, treatment) -> dict[str, float | int]:\n    """Return paired effect estimates and an exact two-sided sign-flip test."""\n    baseline = np.asarray(baseline, dtype=float)\n    treatment = np.asarray(treatment, dtype=float)\n    if baseline.shape != treatment.shape or baseline.ndim != 1 or baseline.size < 2:\n        raise ValueError("baseline and treatment must be paired one-dimensional arrays with n >= 2")\n    if baseline.size > 20:\n        raise ValueError("exact sign-flip enumeration is limited to 20 pairs")\n    differences = treatment - baseline\n    observed = float(differences.mean())\n    permuted = [\n        float(np.mean(differences * np.asarray(signs)))\n        for signs in itertools.product((-1.0, 1.0), repeat=differences.size)\n    ]\n    p_value = float(np.mean(np.abs(permuted) >= abs(observed) - 1e-15))\n    delta_sd = float(np.std(differences, ddof=1))\n    critical = T_CRITICAL_975.get(int(differences.size - 1), math.nan)\n    margin = critical * delta_sd / math.sqrt(differences.size) if not math.isnan(critical) else math.nan\n    return {\n        "paired_n": int(differences.size),\n        "delta_mean": observed,\n        "delta_sd": delta_sd,\n        "delta_ci95_low": observed - margin,\n        "delta_ci95_high": observed + margin,\n        "treatment_better_seeds": int(np.sum(differences > 0)),\n        "treatment_equal_seeds": int(np.sum(differences == 0)),\n        "exact_sign_flip_p_two_sided": p_value,\n    }\n\n\ndef diagnose_public_summary(path: Path | str) -> pd.DataFrame:\n    frame = pd.read_csv(path, encoding="utf-8-sig")\n    required = {\n        "task", "model", "method", "seeds", "f1_mean", "f1_sd",\n        "accuracy_mean", "precision_mean", "recall_mean",\n    }\n    missing = required - set(frame.columns)\n    if missing:\n        raise ValueError(f"summary table is missing columns: {sorted(missing)}")\n\n    rows = []\n    for row in frame.to_dict("records"):\n        num_labels = TASK_NUM_LABELS.get(row["task"])\n        if num_labels is None:\n            raise ValueError(f"unknown task label count: {row[\'task\']}")\n        exact_zero = float(row["f1_sd"]) == 0.0\n        fingerprint = has_constant_prediction_fingerprint(\n            accuracy=float(row["accuracy_mean"]),\n            precision=float(row["precision_mean"]),\n            recall=float(row["recall_mean"]),\n            f1=float(row["f1_mean"]),\n            num_labels=num_labels,\n        )\n        rows.append({\n            "study": row.get("study", ""),\n            "task": row["task"],\n            "model": row["model"],\n            "method": row["method"],\n            "seeds": int(row["seeds"]),\n            "num_labels": num_labels,\n            "accuracy_mean": float(row["accuracy_mean"]),\n            "precision_mean": float(row["precision_mean"]),\n            "recall_mean": float(row["recall_mean"]),\n            "f1_mean": float(row["f1_mean"]),\n            "f1_sd": float(row["f1_sd"]),\n            "f1_sd_exact_zero": exact_zero,\n            "constant_prediction_fingerprint": fingerprint,\n            "degenerate_stability": exact_zero and fingerprint,\n            "diagnosis": (\n                "aggregate fingerprint consistent with constant-class collapse"\n                if exact_zero and fingerprint\n                else "zero variance without constant-class fingerprint"\n                if exact_zero\n                else "non-zero seed variance"\n            ),\n        })\n    return pd.DataFrame(rows)\n\n\ndef summarize_run_frame(frame: pd.DataFrame, expected_seeds: list[int] | None = None) -> pd.DataFrame:\n    required = {\n        "study", "task", "model", "method", "seed", "test_macro_f1",\n        "test_accuracy", "test_macro_precision", "test_macro_recall",\n        "train_seconds", "trainable_params", "trainable_parameter_ratio",\n    }\n    missing = required - set(frame.columns)\n    if missing:\n        raise ValueError(f"run table is missing columns: {sorted(missing)}")\n\n    optional_group_columns = [name for name in ("experiment_id", "variant", "run_mode") if name in frame.columns]\n    group_columns = optional_group_columns + ["study", "task", "model", "method"]\n    rows = []\n    for keys, group in frame.groupby(group_columns, sort=True, dropna=False):\n        if not isinstance(keys, tuple):\n            keys = (keys,)\n        identity = dict(zip(group_columns, keys))\n        seeds = sorted(int(value) for value in group["seed"].tolist())\n        unique_seeds = sorted(set(seeds))\n        f1_values = [float(value) for value in group.sort_values("seed")["test_macro_f1"].tolist()]\n        seed_counts = group["seed"].value_counts()\n        expected = sorted(expected_seeds) if expected_seeds is not None else unique_seeds\n        collapsed = (\n            group["constant_prediction_collapse"].fillna(False).astype(bool)\n            if "constant_prediction_collapse" in group.columns\n            else pd.Series(False, index=group.index)\n        )\n        num_labels = TASK_NUM_LABELS.get(identity["task"])\n        predicted_class_counts = (\n            pd.to_numeric(group["predicted_class_count"], errors="coerce")\n            if "predicted_class_count" in group.columns\n            else pd.Series(math.nan, index=group.index)\n        )\n        majority_rates = (\n            pd.to_numeric(group["majority_prediction_rate"], errors="coerce")\n            if "majority_prediction_rate" in group.columns\n            else pd.Series(math.nan, index=group.index)\n        )\n        near_constant = (\n            group["near_constant_prediction"].fillna(False).astype(bool)\n            if "near_constant_prediction" in group.columns\n            else pd.Series(False, index=group.index)\n        )\n        rows.append({\n            **identity,\n            "method_label": METHOD_LABELS.get(identity["method"], identity["method"]),\n            "runs": len(group),\n            "unique_seed_count": len(unique_seeds),\n            "seeds": json.dumps(unique_seeds, separators=(",", ":")),\n            "missing_seeds": json.dumps(sorted(set(expected) - set(unique_seeds)), separators=(",", ":")),\n            "duplicate_seed_count": int((seed_counts - 1).clip(lower=0).sum()),\n            "f1_values_by_seed": json.dumps(f1_values, separators=(",", ":")),\n            "f1_mean": float(np.mean(f1_values)),\n            "f1_sd": float(np.std(f1_values, ddof=1)) if len(f1_values) > 1 else math.nan,\n            "f1_min": float(np.min(f1_values)),\n            "f1_max": float(np.max(f1_values)),\n            "f1_range": float(np.max(f1_values) - np.min(f1_values)),\n            "f1_all_identical": len(set(f1_values)) == 1,\n            "accuracy_mean": float(group["test_accuracy"].mean()),\n            "precision_mean": float(group["test_macro_precision"].mean()),\n            "recall_mean": float(group["test_macro_recall"].mean()),\n            "train_seconds_mean": float(group["train_seconds"].mean()),\n            "train_seconds_sd": float(group["train_seconds"].std(ddof=1)) if len(group) > 1 else math.nan,\n            "trainable_params_mean": float(group["trainable_params"].mean()),\n            "trainable_ratio_mean": float(group["trainable_parameter_ratio"].mean()),\n            "collapsed_runs": int(collapsed.sum()),\n            "collapse_rate": float(collapsed.mean()),\n            "near_constant_runs": int(near_constant.sum()),\n            "predicted_class_count_min": float(predicted_class_counts.min()),\n            "predicted_class_count_mean": float(predicted_class_counts.mean()),\n            "full_class_coverage_runs": int((predicted_class_counts == num_labels).sum()) if num_labels else 0,\n            "majority_prediction_rate_mean": float(majority_rates.mean()),\n            "majority_prediction_rate_max": float(majority_rates.max()),\n            "train_rows": int(group["train_rows"].iloc[0]) if "train_rows" in group.columns else None,\n            "validation_rows": int(group["validation_rows"].iloc[0]) if "validation_rows" in group.columns else None,\n            "test_rows": int(group["test_rows"].iloc[0]) if "test_rows" in group.columns else None,\n            "epochs_requested": int(group["epochs_requested"].iloc[0]) if "epochs_requested" in group.columns else None,\n            "trainer_devices": json.dumps(sorted(set(str(value) for value in group.get("trainer_device", pd.Series(dtype=str)).dropna())), separators=(",", ":")),\n            "seed_coverage_ok": unique_seeds == expected and len(seeds) == len(unique_seeds),\n        })\n    return pd.DataFrame(rows)\n\n\ndef read_final_metrics(root: Path | str, relative_to: Path | str | None = None) -> pd.DataFrame:\n    rows = []\n    relative_base = Path(relative_to).resolve() if relative_to is not None else None\n    for path in Path(root).glob("**/final_metrics.json"):\n        payload = json.loads(path.read_text(encoding="utf-8"))\n        if payload.get("status") != "COMPLETE":\n            continue\n        resolved_path = path.resolve()\n        payload["source_file"] = str(resolved_path.relative_to(relative_base)) if relative_base else str(resolved_path)\n        rows.append(payload)\n    return pd.DataFrame(rows)\n'
(DRIVE_ROOT / 'src' / 'result_analysis.py').write_text(RESULT_ANALYSIS_SOURCE, encoding='utf-8')
print('result_analysis.py written:', len(RESULT_ANALYSIS_SOURCE), 'chars')


In [ ]:
CONFIG_SOURCE = '{\n  "methods": ["full_ft", "lora", "adapter", "ia3", "bitfit"],\n  "seeds": [42, 52, 62, 72, 82],\n  "max_length": 128,\n  "batch_size": 16,\n  "eval_batch_size": 32,\n  "gradient_accumulation_steps": 4,\n  "precision": "fp16",\n  "weight_decay": 0.01,\n  "warmup_ratio": 0.06,\n  "optimizer": "adamw_torch",\n  "lr_scheduler_type": "linear",\n  "early_stopping_patience": 2,\n  "dataloader_num_workers": 2,\n  "model_revisions": {\n    "FacebookAI/roberta-base": "e2da8e2f811d1448a5b465c236feacd80ffbac7b",\n    "vinai/bertweet-base": "b349c1243407b0dcffeabb2337497477286e27ab"\n  },\n  "learning_rates": {\n    "full_ft": 0.00002,\n    "lora": 0.0001,\n    "adapter": 0.0001,\n    "ia3": 0.0005,\n    "bitfit": 0.0001\n  },\n  "lora": {"r": 8, "alpha": 16, "dropout": 0.05},\n  "adapter": {"bottleneck": 64, "dropout": 0.0},\n  "study1": {"epochs": 3, "models": ["vinai/bertweet-base"], "tasks": ["measuring_hate_speech"]},\n  "study2": {\n    "epochs": 2,\n    "models": ["vinai/bertweet-base", "FacebookAI/roberta-base"],\n    "tasks": ["tweet_sentiment", "finance_sentiment", "movie_reviews", "product_reviews", "tweet_emotion", "tweet_hate", "tweet_offensive", "tweet_irony", "news_topic"],\n    "limits": null\n  },\n  "study3": {"epochs": 5, "models": ["klue/roberta-base"], "tasks": ["news_ynat", "movie_nsmc", "comment_kmhas_binary"]},\n  "smoke_limits": {"train": 128, "validation": 64, "test": 64},\n  "keep_best_checkpoint": true,\n  "continue_on_error": false\n}\n'
(DRIVE_ROOT / 'config' / 'experiment_config.json').write_text(CONFIG_SOURCE, encoding='utf-8')
print(CONFIG_SOURCE)


## 3. A100 및 프로토콜 사전점검

GPU 이름이 A100이 아니면 실행을 중단합니다. 이어서 128/64/64 샘플의 1-epoch smoke run으로 데이터 다운로드부터 테스트 평가까지 실제 동작을 확인합니다. smoke 결과는 Drive에 저장하지 않습니다.


In [ ]:
sys.path.insert(0, str(DRIVE_ROOT))
from src.suite import precheck, load_config, build_jobs, aggregate, run_one, now_iso, atomic_json
import gzip, os, shutil, pandas as pd

PERSIST_RESULTS = PERSIST_ROOT / 'results'
LOCAL_RESULTS = DRIVE_ROOT / 'results'
COMPACT_FILES = ('final_metrics.json', 'status.json', 'run_config.json', 'epoch_metrics.csv', 'trainer_history.csv', 'events.jsonl', 'error.txt')

def restore_compact_results():
    # 마지막 완료 마커가 있는 run만 복원합니다. 동기화 중 끊긴 run은 다음 세션에서 다시 실행됩니다.
    restored = 0
    if PERSIST_RESULTS.exists():
        for marker in PERSIST_RESULTS.glob('study*/PAPER/**/_PERSIST_COMPLETE.json'):
            drive_run = marker.parent
            local_run = LOCAL_RESULTS / drive_run.relative_to(PERSIST_RESULTS)
            local_run.mkdir(parents=True, exist_ok=True)
            for src in drive_run.iterdir():
                if src.is_file() and src.name not in ('predictions.csv.gz', '_PERSIST_COMPLETE.json'):
                    shutil.copy2(src, local_run / src.name)
                    restored += 1
    print('restored compact files:', restored)

def drive_usage_gb():
    total = sum(p.stat().st_size for p in PERSIST_ROOT.rglob('*') if p.is_file())
    return total / 1024**3

def persist_completed_run(job):
    model_slug = job['model_name'].replace('/', '__')
    rel = Path(job['study']) / 'PAPER' / job['task_key'] / model_slug / job['method'] / f"seed_{job['seed']}"
    src_dir, dst_dir = LOCAL_RESULTS / rel, PERSIST_RESULTS / rel
    dst_dir.mkdir(parents=True, exist_ok=True)
    for name in COMPACT_FILES:
        src = src_dir / name
        if src.exists(): shutil.copy2(src, dst_dir / name)
    pred = src_dir / 'predictions.csv'
    if pred.exists():
        local_gz = src_dir / 'predictions.csv.gz'
        with pred.open('rb') as fin, gzip.open(local_gz, 'wb', compresslevel=6) as fout:
            shutil.copyfileobj(fin, fout)
        drive_tmp = dst_dir / 'predictions.csv.gz.tmp'
        shutil.copy2(local_gz, drive_tmp)
        os.replace(drive_tmp, dst_dir / 'predictions.csv.gz')
        local_gz.unlink(missing_ok=True)
    final_metrics = src_dir / 'final_metrics.json'
    if final_metrics.exists():
        atomic_json(dst_dir / '_PERSIST_COMPLETE.json', {'status': 'COMPLETE', 'synced_at': now_iso()})
    shutil.rmtree(src_dir / 'checkpoints', ignore_errors=True)

def run_study_low_drive(study, run_mode='PAPER', max_jobs=None, continue_on_error=False):
    if run_mode != 'PAPER': raise ValueError('이 notebook은 PAPER 모드 전용입니다.')
    jobs = build_jobs(study)
    if max_jobs is not None: jobs = jobs[:max_jobs]
    rows, completed, failed = [], 0, 0
    for index, job in enumerate(jobs, 1):
        try:
            result = run_one(run_mode=run_mode, **job)
            persist_completed_run(job)
            rows.append(result); completed += 1
        except Exception:
            persist_completed_run(job)  # status.json과 error.txt만 보존
            failed += 1
            if not continue_on_error: raise
        progress = {'study': study, 'run_mode': run_mode, 'total': len(jobs), 'index': index, 'completed': completed, 'failed': failed, 'current': job, 'updated_at': now_iso()}
        atomic_json(LOCAL_RESULTS / study / run_mode / 'progress.json', progress)
        atomic_json(PERSIST_RESULTS / study / run_mode / 'progress.json', progress)
        used_gb = drive_usage_gb()
        print(f"[{study}] {index}/{len(jobs)} | compact results {used_gb:.3f} GB")
        if used_gb >= 9.0: raise RuntimeError('저용량 결과 폴더가 9GB에 도달했습니다. 실행을 중단합니다.')
    final = {'study': study, 'run_mode': run_mode, 'status': 'COMPLETE' if failed == 0 else 'COMPLETE_WITH_ERRORS', 'total': len(jobs), 'completed': completed, 'failed': failed, 'updated_at': now_iso()}
    atomic_json(PERSIST_RESULTS / study / run_mode / 'progress.json', final)
    return pd.DataFrame(rows)

def persist_aggregate():
    src = LOCAL_RESULTS / 'aggregate'; dst = PERSIST_RESULTS / 'aggregate'
    if src.exists():
        dst.mkdir(parents=True, exist_ok=True)
        for p in src.iterdir():
            if p.is_file(): shutil.copy2(p, dst / p.name)

free_gb = shutil.disk_usage('/content/drive').free / 1024**3
print(f'Drive free space: {free_gb:.2f} GB')
if free_gb < 0.5: raise RuntimeError('Drive 여유 공간이 0.5GB 미만입니다.')
restore_compact_results()

info = precheck(require_cuda=True)
display(info)
if 'A100' not in info['gpu'].upper():
    raise RuntimeError(f"A100 런타임이 아닙니다: {info['gpu']}")

cfg = load_config()
assert cfg['methods'] == ['full_ft', 'lora', 'adapter', 'ia3', 'bitfit']
assert cfg['seeds'] == [42, 52, 62, 72, 82]
assert cfg['precision'] == 'fp16'
assert cfg['batch_size'] == 16
assert cfg['gradient_accumulation_steps'] == 4
assert cfg['study2']['limits'] is None
print('PROTOCOL CHECK PASS')
print('Study 1 jobs:', len(build_jobs('study1')))
print('Study 2 jobs:', len(build_jobs('study2')))
print('Study 3 jobs:', len(build_jobs('study3')))
assert sum(len(build_jobs(s)) for s in ('study1', 'study2', 'study3')) == 550

RUN_END_TO_END_SMOKE = True
if RUN_END_TO_END_SMOKE:
    smoke_job = build_jobs('study1')[0]
    print('Starting end-to-end Colab smoke test:', smoke_job['model_name'], smoke_job['method'])
    smoke_result = run_one(run_mode='SMOKE', **smoke_job)
    assert smoke_result['status'] == 'COMPLETE'
    assert 'test_macro_f1' in smoke_result
    print('END-TO-END SMOKE PASS | test_macro_f1 =', smoke_result['test_macro_f1'])
    shutil.rmtree(LOCAL_RESULTS / 'study1' / 'SMOKE', ignore_errors=True)


## 4. Study 1 실행 — 25 Runs

BERTweet + Measuring Hate Speech, 5 methods × 5 seeds. 예상 A100 시간은 로컬보다 짧지만 Drive 저장시간에 따라 달라질 수 있습니다.


In [ ]:
STUDY = 'study1'
RUN_MODE = 'PAPER'
study1_result = run_study_low_drive(STUDY, run_mode=RUN_MODE, max_jobs=None, continue_on_error=False)
display(study1_result.tail())


## 5. Study 3 실행 — 75 Runs

KLUE-RoBERTa + YNAT/NSMC/K-MHaS, 5 methods × 5 seeds.


In [ ]:
STUDY = 'study3'
RUN_MODE = 'PAPER'
study3_result = run_study_low_drive(STUDY, run_mode=RUN_MODE, max_jobs=None, continue_on_error=False)
display(study3_result.tail())


## 6. Study 2 실행 — 450 Runs

9 English tasks × 2 models × 5 methods × 5 seeds. 원본 split 전체를 사용합니다.


In [ ]:
STUDY = 'study2'
RUN_MODE = 'PAPER'
study2_result = run_study_low_drive(STUDY, run_mode=RUN_MODE, max_jobs=None, continue_on_error=False)
display(study2_result.tail())


## 7. 전체 결과 집계

세 Study가 완료된 뒤 실행합니다.


In [ ]:
all_runs = aggregate('PAPER')
persist_aggregate()
display(all_runs.head())
print('completed result rows:', len(all_runs), '/ 550')
assert len(all_runs) == 550, '아직 완료되지 않은 run이 있습니다.'


## 8. 진행상태 확인

학습 셀이 중단된 뒤 또는 Study 사이에 실행할 수 있습니다.


In [ ]:
from collections import Counter

for study in ('study1', 'study3', 'study2'):
    root = DRIVE_ROOT / 'results' / study / 'PAPER'
    statuses = []
    for path in root.glob('**/status.json') if root.exists() else []:
        try:
            statuses.append(json.loads(path.read_text(encoding='utf-8')).get('status', 'UNKNOWN'))
        except Exception:
            statuses.append('UNREADABLE')
    print(study, dict(Counter(statuses)))
    progress = root / 'progress.json'
    if progress.exists():
        print(progress.read_text(encoding='utf-8'))


## 재개 방법과 저장 정책

1. Colab 연결이 끊기면 1~3번 셀을 다시 실행합니다.
2. 중단된 Study의 실행 셀을 다시 실행합니다.
3. `COMPLETE` run은 자동으로 건너뜁니다.
4. 중간 checkpoint를 Drive에 저장하지 않으므로 중단 당시 실행 중이던 1개 run만 처음부터 다시 시작합니다.
5. 완료 run마다 `final_metrics.json`, 설정, 학습 로그, gzip 예측 파일을 Drive에 저장합니다.
6. 모델 checkpoint와 Hugging Face dataset/model cache는 `/content`에만 존재하며 완료 즉시 checkpoint를 삭제합니다.
7. 각 run 후 출력되는 `Drive 0.000 GB` 값을 확인하여 10GB 한도를 모니터링합니다.
8. 오류 발생 시 Drive의 해당 run 폴더에서 `error.txt`와 `status.json`을 확인합니다.
